# StudentCup2026 — H214 Prediction Pipeline

企業データから **DX教育商材の購入有無** を予測する実行用ノートブックです。



## 1. Import

In [ ]:
import json
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")

## 2. Configuration

パス、列名、モデルのハイパーパラメータ、後処理をここでまとめて管理します。

In [ ]:
@dataclass
class Config:
    # ---------- Path ----------
    DATA_DIR: Optional[str] = None
    SUBMISSION_DIR: str = "/content/drive/MyDrive/StudentCup2026/submissions"
    OUTPUT_DIR: str = "/content/drive/MyDrive/StudentCup2026/artifacts"
    AUTO_MOUNT_DRIVE: bool = True

    # ---------- Columns ----------
    ID_COL: str = "企業ID"
    TARGET_COL: str = "購入フラグ"
    COMPANY_NAME_COL: str = "企業名"
    TEXT_COLUMNS: tuple = ("企業概要", "組織図", "今後のDX展望")

    # ---------- Text model ----------
    HASH_FEATURES: int = 32768
    NGRAM_RANGE: tuple = (2, 5)
    SVC_C: float = 0.3

    # ---------- Structured model ----------
    LR_C: float = 2.0

    # ---------- Bagging / Blend ----------
    SEEDS: tuple = (42,)
    N_SPLITS: int = 5
    N_JOBS: int = 4
    H214_BLEND_WEIGHT: float = 0.50
    H214_THRESHOLD: float = 0.28

    # ---------- Postprocessing ----------
    APPLY_PUBLIC_BEST_OVERRIDE: bool = True
    PUBLIC_BEST_VETO_IDS: tuple = (1094,)

    # 棄却済み実験。通常はFalseのまま使用する。
    APPLY_REJECTED_RECALL_RESCUE: bool = False
    RECALL_LOWER: float = 0.10
    RECALL_UPPER: float = 0.28

    # ---------- Output ----------
    SUBMISSION_NAME: str = "submission_H214_current_best_noheader.csv"
    PROBABILITY_NAME: str = "H214_test_probabilities_60seed.csv"  # 既存出力名を維持
    CHANGE_LOG_NAME: str = "H214_change_log.csv"
    RUN_INFO_NAME: str = "H214_run_info.json"

    # ---------- EDA ----------
    RUN_EDA: bool = True


CONFIG = Config()
EPS = 1e-9
REQUIRED_FILES = ("train.csv", "test.csv")

## 3. Data Loading

Google Driveのmount、データディレクトリ探索、CSV読み込み、入力チェックを担当します。

In [ ]:
def maybe_mount_drive() -> None:
    """Colab環境の場合のみGoogle Driveをmountする。"""
    if not CONFIG.AUTO_MOUNT_DRIVE:
        return

    try:
        from google.colab import drive  # type: ignore
    except Exception:
        return

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")


def resolve_data_dir(explicit_dir: Optional[str] = None) -> Path:
    """train.csv と test.csv が存在するディレクトリを返す。"""
    candidates = []
    if explicit_dir:
        candidates.append(Path(explicit_dir).expanduser())

    candidates.extend([
        Path.cwd(),
        Path.cwd() / "data",
        Path("/content"),
        Path("/content/data"),
        Path("/content/drive/MyDrive/StudentCup2026/data"),
        Path("/mnt/data"),
    ])

    checked = []
    for directory in candidates:
        try:
            directory = directory.resolve()
        except Exception:
            directory = Path(directory)

        if directory in checked:
            continue
        checked.append(directory)

        if all((directory / name).exists() for name in REQUIRED_FILES):
            return directory

    checked_text = "\n".join(f"- {p}" for p in checked)
    raise FileNotFoundError(
        "train.csv / test.csv が見つかりません。CONFIG.DATA_DIRを設定してください。"
        f"\n\n確認した場所:\n{checked_text}"
    )


def prepare_paths():
    """入力・提出・artifactの保存先を準備する。"""
    maybe_mount_drive()

    data_dir = resolve_data_dir(CONFIG.DATA_DIR)
    submission_dir = Path(CONFIG.SUBMISSION_DIR).expanduser()
    output_dir = Path(CONFIG.OUTPUT_DIR).expanduser()

    drive_available = Path("/content/drive/MyDrive").exists()
    if str(submission_dir).startswith("/content/drive") and not drive_available:
        submission_dir = data_dir / "generated_h214_submissions"
    if str(output_dir).startswith("/content/drive") and not drive_available:
        output_dir = data_dir / "generated_h214_artifacts"

    submission_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, submission_dir, output_dir


def validate_input_data(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    """学習・予測に必要な最低限の列を確認する。"""
    required_train = {CONFIG.ID_COL, CONFIG.COMPANY_NAME_COL, CONFIG.TARGET_COL}
    required_test = {CONFIG.ID_COL, CONFIG.COMPANY_NAME_COL}

    missing_train = required_train - set(train_df.columns)
    missing_test = required_test - set(test_df.columns)

    if missing_train:
        raise ValueError(f"trainに必要列がありません: {sorted(missing_train)}")
    if missing_test:
        raise ValueError(f"testに必要列がありません: {sorted(missing_test)}")


def load_competition_data(data_dir: Path):
    """train / test / submission templateを読み込む。"""
    train_df = pd.read_csv(data_dir / "train.csv")
    test_df = pd.read_csv(data_dir / "test.csv")
    validate_input_data(train_df, test_df)

    sample_path = data_dir / "sample_submit.csv"
    if sample_path.exists():
        sample_submit_df = pd.read_csv(sample_path, header=None)
        template_source = "sample_submit.csv"
    else:
        sample_submit_df = pd.DataFrame({
            0: test_df[CONFIG.ID_COL].values,
            1: np.zeros(len(test_df), dtype=int),
        })
        template_source = "generated_from_test_id"

    if len(sample_submit_df) != len(test_df) or sample_submit_df.shape[1] < 2:
        raise ValueError(
            f"提出テンプレート形状が不正です: {sample_submit_df.shape}, "
            f"test={test_df.shape}"
        )

    return train_df, test_df, sample_submit_df, template_source

## 4. EDA


In [ ]:
def run_basic_eda(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    """shape、目的変数、型、欠損率などを簡潔に確認する。"""
    print("=== Shape ===")
    print("train:", train_df.shape)
    print("test :", test_df.shape)

    print("\n=== Target ===")
    target_count = train_df[CONFIG.TARGET_COL].value_counts().sort_index()
    target_ratio = train_df[CONFIG.TARGET_COL].value_counts(normalize=True).sort_index()
    target_summary = pd.DataFrame({"count": target_count, "ratio": target_ratio})
    display(target_summary)

    print("=== Dtype summary ===")
    dtype_summary = train_df.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("count")
    display(dtype_summary)

    print("=== Missing rate: top 15 ===")
    missing = (
        train_df.isna().mean()
        .sort_values(ascending=False)
        .head(15)
        .rename("missing_rate")
        .to_frame()
    )
    display(missing)

    train_only = sorted(set(train_df.columns) - set(test_df.columns) - {CONFIG.TARGET_COL})
    test_only = sorted(set(test_df.columns) - set(train_df.columns))
    print("train only columns (except target):", train_only)
    print("test only columns:", test_only)

## 5. Feature Engineering / Preprocessing

### 5.1 Feature Engineering Utilities

In [ ]:
EDUCATION_TERMS = ("教育", "研修")
INVESTMENT_TERMS = ("投資",)
POSITIVE_CONTEXT_TERMS = (
    "拡充", "強化", "拡大", "増加", "充実", "積極", "加速", "推進", "底上げ",
    "重点", "増強", "全面", "大幅", "投下", "刷新", "高度化", "実践", "拡張",
)
CAUTIOUS_CONTEXT_TERMS = (
    "抑制", "見送り", "必要最低限", "最低限", "限定", "慎重", "縮小", "保守",
    "維持", "現状維持", "選択的", "絞", "段階的", "控え", "据え置", "行わない",
    "実施しない", "予定していない", "予定しておりません",
)
FUTURE_MARKERS = ("今後", "次期", "これから", "次のフェーズ", "将来")
FIXED_DROPPED_FEATURES = (
    "log__営業利益率",
    "log__今後のDX展望__行数",
    "log__今後のDX展望__kw_デジタル",
)


def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    denominator = b.astype(float).where(b.abs() > EPS, np.nan)
    return a.astype(float) / denominator


def signed_log1p(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    return np.sign(values) * np.log1p(np.abs(values))


def categorical_text(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("欠損").astype(str)


def extract_future_section(text: str) -> str:
    text = "" if text is None else str(text)
    if not text:
        return ""

    start = int(len(text) * 0.20)
    search_text = text[start:]
    positions = []
    for marker in FUTURE_MARKERS:
        pos = search_text.find(marker)
        if pos >= 0:
            positions.append(start + pos)

    if positions:
        return text[min(positions):]
    return text[int(len(text) * 0.60):]


def count_terms(text: str, terms: tuple) -> float:
    return float(sum(text.count(term) for term in terms))


def split_japanese_sentences(text: str) -> list:
    if not text:
        return []
    return [
        sentence.strip()
        for sentence in re.split(r"[。！？!?]+|\n+", str(text))
        if sentence and sentence.strip()
    ]


def contains_any(sentence: str, terms: tuple) -> bool:
    return any(term in sentence for term in terms)


def make_dx_context_features(series: pd.Series) -> pd.DataFrame:
    """DX展望内の教育・投資に関する積極/慎重文脈を数値化する。"""
    records = []
    for value in series.fillna("").astype(str):
        sentences = split_japanese_sentences(value)
        education_sentences = [s for s in sentences if contains_any(s, EDUCATION_TERMS)]
        investment_sentences = [s for s in sentences if contains_any(s, INVESTMENT_TERMS)]

        records.append({
            "ctx__DX展望__教育関連文数": float(len(education_sentences)),
            "ctx__DX展望__教育_積極文数": float(sum(contains_any(s, POSITIVE_CONTEXT_TERMS) for s in education_sentences)),
            "ctx__DX展望__教育_慎重文数": float(sum(contains_any(s, CAUTIOUS_CONTEXT_TERMS) for s in education_sentences)),
            "ctx__DX展望__投資_積極文数": float(sum(contains_any(s, POSITIVE_CONTEXT_TERMS) for s in investment_sentences)),
            "ctx__DX展望__投資_慎重文数": float(sum(contains_any(s, CAUTIOUS_CONTEXT_TERMS) for s in investment_sentences)),
        })

    return pd.DataFrame(records, index=series.index)

### 5.2 Feature Engineering

In [ ]:
def add_engineered_features(
    df: pd.DataFrame,
    *,
    include_soft_sales: bool = True,
    add_q4_interaction: bool = False,
) -> pd.DataFrame:
    """構造化モデル用の派生特徴量を生成する。"""
    out = df.copy()

    # --- Financial ratios ---
    ratio_defs = {
        "自己資本比率": ("自己資本", "総資産"),
        "流動資産比率": ("流動資産", "総資産"),
        "固定資産比率": ("固定資産", "総資産"),
        "負債比率": ("負債", "総資産"),
        "借入金比率": (None, "総資産"),
        "営業利益率": ("営業利益", "売上"),
        "経常利益率": ("経常利益", "売上"),
        "純利益率": ("当期純利益", "売上"),
        "ROA": ("当期純利益", "総資産"),
        "ROE": ("当期純利益", "自己資本"),
        "営業CFマージン": ("営業CF", "売上"),
        "投資CF対売上": ("投資CF", "売上"),
        "売上高総資産回転率": ("売上", "総資産"),
        "一人当たり売上": ("売上", "従業員数"),
        "一人当たり純利益": ("当期純利益", "従業員数"),
    }

    if {"短期借入金", "長期借入金", "総資産"}.issubset(out.columns):
        out["借入金合計"] = out["短期借入金"] + out["長期借入金"]

    for feature_name, (numerator, denominator) in ratio_defs.items():
        if feature_name == "借入金比率":
            if {"借入金合計", denominator}.issubset(out.columns):
                out[feature_name] = safe_divide(out["借入金合計"], out[denominator])
        elif numerator in out.columns and denominator in out.columns:
            out[feature_name] = safe_divide(out[numerator], out[denominator])

    # --- Software investment ---
    software_col = "無形固定資産変動(ソフトウェア関連)"
    if {software_col, "投資CF"}.issubset(out.columns):
        out["ソフトウェア投資比率"] = safe_divide(
            out[software_col].abs(), out["投資CF"].abs()
        )

    if include_soft_sales and {software_col, "売上"}.issubset(out.columns):
        out["soft_sales"] = safe_divide(out[software_col].abs(), out["売上"].abs())

    if add_q4_interaction and "soft_sales" in out.columns and "アンケート４" in out.columns:
        q4 = pd.to_numeric(out["アンケート４"], errors="coerce")
        out["soft_x_q4"] = out["soft_sales"] * q4

    # --- Text-derived numeric features ---
    keywords = (
        "DX", "AI", "IoT", "クラウド", "研修", "教育", "投資", "積極", "前のめり",
        "慎重", "段階的", "効率化", "デジタル",
    )
    digit_drop = {"企業概要", "今後のDX展望"}

    for col in [c for c in CONFIG.TEXT_COLUMNS if c in out.columns]:
        text = out[col].fillna("").astype(str)
        prefix = f"{col}__"
        out[prefix + "文字数"] = text.str.len().astype(float)
        out[prefix + "行数"] = text.str.count(r"\n").add(1).astype(float)
        if col not in digit_drop:
            out[prefix + "数字数"] = text.str.count(r"\d").astype(float)
        out[prefix + "記号数"] = text.str.count(r"[^\w\s]").astype(float)
        for keyword in keywords:
            out[prefix + f"kw_{keyword}"] = text.str.count(keyword).astype(float)

    # --- DX context ---
    if "今後のDX展望" in out.columns:
        context_df = make_dx_context_features(out["今後のDX展望"])
        for col in context_df.columns:
            out[col] = context_df[col]

        future_sections = out["今後のDX展望"].fillna("").astype(str).map(extract_future_section)
        out["future__DX展望__慎重語数"] = future_sections.map(
            lambda text: count_terms(text, CAUTIOUS_CONTEXT_TERMS)
        ).astype(float)

    # --- Company size × DX text length ---
    if "従業員数" in out.columns and "今後のDX展望" in out.columns:
        employees = pd.to_numeric(out["従業員数"], errors="coerce")
        dx_length = out["今後のDX展望"].fillna("").astype(str).str.len()

        size_bin = np.select(
            [employees <= 1000, (employees > 1000) & (employees <= 2000), employees > 2000],
            ["small", "mid", "large"],
            default="missing",
        )
        dx_bin = np.where(dx_length <= 750, "short", "long")
        out["cat__size_x_dxlen"] = (
            pd.Series(size_bin, index=out.index).astype(str)
            + "__"
            + pd.Series(dx_bin, index=out.index).astype(str)
        )

    # --- Facility missingness × industry ---
    for col in ("工場数", "店舗数"):
        if col not in out.columns:
            continue

        state_col = f"cat__{col}_入力状態"
        out[state_col] = np.where(out[col].isna(), "欠損", "入力あり")

        if "業界" in out.columns:
            out[f"cat__業界×{col}_入力状態"] = (
                categorical_text(out["業界"]) + "__×__" + out[state_col]
            )

    # --- Signed log features ---
    excluded = {CONFIG.ID_COL, CONFIG.TARGET_COL}
    numeric_cols = [
        col for col in out.select_dtypes(include=[np.number]).columns
        if col not in excluded
        and not col.startswith("アンケート")
        and not col.startswith("miss__")
    ]
    for col in numeric_cols:
        out[f"log__{col}"] = signed_log1p(out[col])

    # --- Missing flags ---
    for col in ("アンケート７", "事業所数", "資本金", "営業利益", "経常利益"):
        if col in out.columns:
            out[f"miss__{col}"] = out[col].isna().astype(np.float32)

    return out.replace([np.inf, -np.inf], np.nan)

### 5.3 Structured Preprocessor

In [ ]:
def make_onehot_encoder() -> OneHotEncoder:
    kwargs = dict(handle_unknown="ignore", min_frequency=2, dtype=np.float32)
    try:
        return OneHotEncoder(sparse_output=True, **kwargs)
    except TypeError:
        return OneHotEncoder(sparse=True, **kwargs)


class FeatureBuilder:
    """特徴量生成と数値/カテゴリ前処理をまとめるクラス。"""

    def __init__(self, *, include_soft_sales: bool = True, add_q4_interaction: bool = False):
        self.include_soft_sales = include_soft_sales
        self.add_q4_interaction = add_q4_interaction

    def fit(self, df: pd.DataFrame):
        engineered = add_engineered_features(
            df,
            include_soft_sales=self.include_soft_sales,
            add_q4_interaction=self.add_q4_interaction,
        )

        drop_cols = {CONFIG.TARGET_COL, CONFIG.ID_COL, CONFIG.COMPANY_NAME_COL}
        text_cols = [c for c in CONFIG.TEXT_COLUMNS if c in engineered.columns]

        object_cols = list(engineered.select_dtypes(include=["object", "category"]).columns)
        self.cat_cols = [c for c in object_cols if c not in set(text_cols) | drop_cols]

        numeric_cols = [
            c for c in engineered.select_dtypes(include=[np.number]).columns
            if c not in drop_cols
        ]
        survey_cols = [c for c in numeric_cols if c.startswith("アンケート")]
        log_cols = [c for c in numeric_cols if c.startswith("log__")]
        missing_cols = [c for c in numeric_cols if c.startswith("miss__")]

        self.num_cols = list(dict.fromkeys(survey_cols + log_cols + missing_cols))
        self.num_cols = [c for c in self.num_cols if c not in FIXED_DROPPED_FEATURES]

        self.num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler(with_centering=False)),
        ])
        self.num_pipeline.fit(engineered[self.num_cols])

        self.cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ])
        if self.cat_cols:
            self.cat_pipeline.fit(engineered[self.cat_cols])

        return self

    def transform(self, df: pd.DataFrame):
        engineered = add_engineered_features(
            df,
            include_soft_sales=self.include_soft_sales,
            add_q4_interaction=self.add_q4_interaction,
        )

        blocks = [
            csr_matrix(
                self.num_pipeline.transform(engineered[self.num_cols]),
                dtype=np.float32,
            )
        ]

        if self.cat_cols:
            blocks.append(
                csr_matrix(
                    self.cat_pipeline.transform(engineered[self.cat_cols]),
                    dtype=np.float32,
                )
            )

        return hstack(blocks, format="csr", dtype=np.float32)

    def fit_transform(self, df: pd.DataFrame):
        return self.fit(df).transform(df)

## 6. Modeling

### 6.1 Text Model

In [ ]:
def make_text_series(df: pd.DataFrame) -> np.ndarray:
    """3つの自由記述列を1つのテキストへ結合する。"""
    return (
        df[list(CONFIG.TEXT_COLUMNS)]
        .fillna("")
        .astype(str)
        .agg("\n".join, axis=1)
        .values
    )


def make_hashing_vectorizer() -> HashingVectorizer:
    return HashingVectorizer(
        analyzer="char",
        ngram_range=CONFIG.NGRAM_RANGE,
        n_features=CONFIG.HASH_FEATURES,
        alternate_sign=False,
        norm=None,
        dtype=np.float32,
    )


def sigmoid(values: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(values, -40, 40)))


def logit(probability: np.ndarray) -> np.ndarray:
    probability = np.clip(probability, 1e-8, 1 - 1e-8)
    return np.log(probability / (1 - probability))

### 6.2 H214 Training / Inference

In [ ]:
def fit_repeated_probabilities(
    X: pd.DataFrame,
    y: np.ndarray,
    test_X: pd.DataFrame,
    *,
    need_q4: bool = True,
):
    """各seedで80% subset学習し、H196 / Q4 branchの予測確率を平均する。"""
    hashing = make_hashing_vectorizer()
    hashed_train = hashing.transform(make_text_series(X))
    hashed_test = hashing.transform(make_text_series(test_X))

    def train_one_seed(seed: int):
        # 5-foldの最初のtraining fold (= 約80%) をbagging subsetとして使用
        splitter = StratifiedKFold(
            n_splits=CONFIG.N_SPLITS,
            shuffle=True,
            random_state=seed,
        )
        train_idx, _ = next(splitter.split(X, y))

        # ----- Text branch -----
        tfidf = TfidfTransformer(sublinear_tf=True)
        text_train = tfidf.fit_transform(hashed_train[train_idx])
        text_test = tfidf.transform(hashed_test)

        text_model = LinearSVC(C=CONFIG.SVC_C, random_state=seed)
        text_model.fit(text_train, y[train_idx])
        text_score = text_model.decision_function(text_test)

        # ----- Base structured branch -----
        base_builder = FeatureBuilder(
            include_soft_sales=True,
            add_q4_interaction=False,
        )
        struct_train = base_builder.fit_transform(X.iloc[train_idx])
        struct_test = base_builder.transform(test_X)

        base_model = LogisticRegression(
            C=CONFIG.LR_C,
            solver="liblinear",
            max_iter=5000,
            random_state=seed,
        )
        base_model.fit(struct_train, y[train_idx])
        base_struct_prob = base_model.predict_proba(struct_test)[:, 1]
        base_prob = sigmoid(logit(base_struct_prob) + text_score)

        if not need_q4:
            return base_prob, None

        # ----- Q4 interaction branch -----
        q4_builder = FeatureBuilder(
            include_soft_sales=True,
            add_q4_interaction=True,
        )
        q4_train = q4_builder.fit_transform(X.iloc[train_idx])
        q4_test = q4_builder.transform(test_X)

        q4_model = LogisticRegression(
            C=CONFIG.LR_C,
            solver="liblinear",
            max_iter=5000,
            random_state=seed,
        )
        q4_model.fit(q4_train, y[train_idx])
        q4_struct_prob = q4_model.predict_proba(q4_test)[:, 1]
        q4_prob = sigmoid(logit(q4_struct_prob) + text_score)

        return base_prob, q4_prob

    results = Parallel(n_jobs=CONFIG.N_JOBS, verbose=10)(
        delayed(train_one_seed)(seed) for seed in CONFIG.SEEDS
    )

    base_matrix = np.vstack([result[0] for result in results])
    h196_prob = base_matrix.mean(axis=0)

    if not need_q4:
        return h196_prob, None, base_matrix

    q4_matrix = np.vstack([result[1] for result in results])
    q4_prob = q4_matrix.mean(axis=0)
    return h196_prob, q4_prob, base_matrix


def compute_h214_probabilities(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> np.ndarray:
    """train/testからH214予測確率を計算する。"""
    y = train_df[CONFIG.TARGET_COL].astype(int).to_numpy()

    X = train_df.drop(
        columns=[CONFIG.TARGET_COL, CONFIG.ID_COL, CONFIG.COMPANY_NAME_COL]
    )
    test_X = test_df.drop(
        columns=[CONFIG.ID_COL, CONFIG.COMPANY_NAME_COL]
    )

    h196_prob, q4_prob, _ = fit_repeated_probabilities(
        X,
        y,
        test_X,
        need_q4=True,
    )

    if q4_prob is None:
        raise RuntimeError("Q4 branch probability was not generated.")

    return (
        CONFIG.H214_BLEND_WEIGHT * h196_prob
        + (1.0 - CONFIG.H214_BLEND_WEIGHT) * q4_prob
    )

## 7. Prediction / Postprocessing

閾値判定と、現在の提出を再現するための後処理を分離しています。

In [ ]:
def apply_prediction_rules(
    test_df: pd.DataFrame,
    h214_prob: np.ndarray,
):
    pred = (h214_prob >= CONFIG.H214_THRESHOLD).astype(int)
    change_log = []

    # Public leaderboard診断由来の後処理
    if CONFIG.APPLY_PUBLIC_BEST_OVERRIDE:
        veto_mask = test_df[CONFIG.ID_COL].isin(CONFIG.PUBLIC_BEST_VETO_IDS).to_numpy()
        changed_idx = np.where(veto_mask & (pred == 1))[0]

        for idx in changed_idx:
            change_log.append({
                CONFIG.ID_COL: int(test_df.iloc[idx][CONFIG.ID_COL]),
                "change": "1->0",
                "reason": "public_best_override",
            })
        pred[veto_mask] = 0

    # 棄却済み実験の再現用。通常はOFF。
    if CONFIG.APPLY_REJECTED_RECALL_RESCUE:
        q2 = pd.to_numeric(test_df["アンケート２"], errors="coerce")
        q4 = pd.to_numeric(test_df["アンケート４"], errors="coerce")
        q10 = pd.to_numeric(test_df["アンケート１０"], errors="coerce")

        rescue_mask = (
            (h214_prob >= CONFIG.RECALL_LOWER)
            & (h214_prob < CONFIG.RECALL_UPPER)
            & (q4.to_numpy() == 2)
            & (q2.to_numpy() < 5)
            & (q10.to_numpy() >= 3)
        )

        changed_idx = np.where(rescue_mask & (pred == 0))[0]
        for idx in changed_idx:
            change_log.append({
                CONFIG.ID_COL: int(test_df.iloc[idx][CONFIG.ID_COL]),
                "change": "0->1",
                "reason": "rejected_recall_rescue_reproduction",
            })
        pred[rescue_mask] = 1

    return pred, change_log


def build_submission(
    sample_submit_df: pd.DataFrame,
    test_df: pd.DataFrame,
    pred: np.ndarray,
) -> pd.DataFrame:
    """sample_submitのID順を保持した2列submissionを作る。"""
    pred_by_id = pd.Series(
        pred.astype(int),
        index=test_df[CONFIG.ID_COL].astype(int).to_numpy(),
    )

    submission = sample_submit_df.iloc[:, :2].copy()
    submission_ids = pd.to_numeric(submission.iloc[:, 0], errors="raise").astype(int)

    missing_ids = [x for x in submission_ids if x not in pred_by_id.index]
    if missing_ids:
        raise ValueError(f"sample_submit内にtestに存在しないIDがあります: {missing_ids[:10]}")

    submission.iloc[:, 1] = submission_ids.map(pred_by_id).astype(int).to_numpy()
    return submission

## 8. Execution


In [ ]:
def main():
    started = time.time()

    # 1. Path / Data
    data_dir, submission_dir, output_dir = prepare_paths()
    train_df, test_df, sample_submit_df, template_source = load_competition_data(data_dir)

    print("DATA_DIR      :", data_dir)
    print("SUBMISSION_DIR:", submission_dir)
    print("OUTPUT_DIR    :", output_dir)
    print("SEEDS         :", CONFIG.SEEDS)
    print("N_JOBS        :", CONFIG.N_JOBS)

    # 2. EDA
    if CONFIG.RUN_EDA:
        print("\n[EDA]")
        run_basic_eda(train_df, test_df)

    # 3. Prediction
    print("\n[Prediction] H214 probability calculation...")
    h214_prob = compute_h214_probabilities(train_df, test_df)

    # 4. Save probabilities
    probability_df = pd.DataFrame({
        CONFIG.ID_COL: test_df[CONFIG.ID_COL].astype(int),
        "p_h214": h214_prob,
        "h214_pred_raw": (h214_prob >= CONFIG.H214_THRESHOLD).astype(int),
    })

    for col in ("アンケート２", "アンケート４", "アンケート１０"):
        if col in test_df.columns:
            probability_df[col] = test_df[col].values

    probability_path = output_dir / CONFIG.PROBABILITY_NAME
    probability_df.to_csv(probability_path, index=False, encoding="utf-8-sig")

    # 5. Postprocessing
    pred, change_log = apply_prediction_rules(test_df, h214_prob)

    # 6. Submission
    submission = build_submission(sample_submit_df, test_df, pred)
    submission_path = submission_dir / CONFIG.SUBMISSION_NAME
    submission.to_csv(
        submission_path,
        index=False,
        header=False,
        encoding="utf-8-sig",
    )

    # 7. Logs / Metadata
    change_log_path = output_dir / CONFIG.CHANGE_LOG_NAME
    pd.DataFrame(
        change_log,
        columns=[CONFIG.ID_COL, "change", "reason"],
    ).to_csv(change_log_path, index=False, encoding="utf-8-sig")

    run_info = {
        "data_dir": str(data_dir),
        "submission_dir": str(submission_dir),
        "output_dir": str(output_dir),
        "template_source": template_source,
        "train_shape": list(train_df.shape),
        "test_shape": list(test_df.shape),
        "seeds": list(CONFIG.SEEDS),
        "n_jobs": CONFIG.N_JOBS,
        "h214_threshold": CONFIG.H214_THRESHOLD,
        "raw_h214_positive": int((h214_prob >= CONFIG.H214_THRESHOLD).sum()),
        "final_positive": int(pred.sum()),
        "apply_public_best_override": CONFIG.APPLY_PUBLIC_BEST_OVERRIDE,
        "public_best_veto_ids": list(CONFIG.PUBLIC_BEST_VETO_IDS),
        "apply_rejected_recall_rescue": CONFIG.APPLY_REJECTED_RECALL_RESCUE,
        "change_count": len(change_log),
        "elapsed_seconds": round(time.time() - started, 2),
        "probability_file": str(probability_path),
        "submission_file": str(submission_path),
        "change_log_file": str(change_log_path),
    }

    run_info_path = output_dir / CONFIG.RUN_INFO_NAME
    with run_info_path.open("w", encoding="utf-8") as file:
        json.dump(run_info, file, ensure_ascii=False, indent=2)

    print("\n=== DONE ===")
    print("raw H214 positive:", run_info["raw_h214_positive"])
    print("final positive   :", run_info["final_positive"])
    print("changes          :", len(change_log))
    print("probabilities    :", probability_path)
    print("change log       :", change_log_path)
    print("run info         :", run_info_path)
    print("submission       :", submission_path)

    return {
        "probabilities": probability_df,
        "submission": submission,
        "change_log": pd.DataFrame(change_log),
        "run_info": run_info,
    }

### Run

In [ ]:
results = main()

## 9. Notes

- **EDAはモデル学習と分離**しています。不要な場合は `CONFIG.RUN_EDA = False` にしてください。
- **Public Best Overrideはモデル本体とは別の後処理**です。純粋なモデル性能を確認する場合は `CONFIG.APPLY_PUBLIC_BEST_OVERRIDE = False` にしてください。
- 現在の `CONFIG.SEEDS` は `(42,)` のため、実際の実行は1-seedです。過去の「60-seed」表記とは一致していません。
- 過去モデル（H28/H98等）の再現用関数は、現在のH214実行には不要なため、この実行用Notebookからは外しています。